# 06 Market Basket Analysis

Apriori finds product co occurrence rules of the form `{A, B} => {C}` with support, confidence, and lift. The classical use case requires multi item invoices.

This dataset records one `Product line` per invoice, so a per invoice basket would be a single item set and Apriori would have nothing to mine. The workaround is to **aggregate by `Branch` and `Date`**: each row of the basket matrix is one branch day, and the columns are 1 if that product line sold that day, 0 otherwise. The resulting rules read as: on a typical branch day, when X and Y sell, Z is also likely to sell. This is enough to drive co location and bundling recommendations even without a true item level basket.

Support thresholds are scanned from 0.20 down to 0.03. The first threshold that produces at least one rule is kept, so output exists even on sparse branches. Rules are filtered by `lift > 1` (positive association) and ranked by lift to surface the strongest co movements first.

In [1]:
import os, warnings
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')
os.environ.setdefault('TQDM_DISABLE', '1')
warnings.filterwarnings('ignore')
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.viz import set_style, fmt_money

set_style()
RNG = np.random.default_rng(config.RANDOM_SEED)


In [2]:
from src.data_loader import load_processed
from mlxtend.frequent_patterns import apriori, association_rules

df = load_processed()


## Build daily branch baskets


In [3]:
baskets = (
    df.assign(flag=1)
      .pivot_table(index=['Branch', 'Date'], columns='Product line', values='flag', aggfunc='max', fill_value=0)
)
baskets.head()


Product line       Electronic accessories  Fashion accessories  \
Branch Date                                                      
A      2019-01-01                       1                    1   
       2019-01-02                       0                    0   
       2019-01-03                       0                    0   
       2019-01-04                       0                    0   
       2019-01-05                       1                    0   

Product line       Food and beverages  Health and beauty  Home and lifestyle  \
Branch Date                                                                    
A      2019-01-01                   0                  0                   1   
       2019-01-02                   1                  0                   0   
       2019-01-03                   0                  1                   1   
       2019-01-04                   0                  1                   1   
       2019-01-05                   0                  1                   1   

Product line       Sports and travel  
Branch Date                           
A      2019-01-01                  1  
       2019-01-02                  1  
       2019-01-03                  0  
       2019-01-04                  0  
       2019-01-05                  0

## Run Apriori
We scan a small grid of support thresholds and keep the lowest threshold that yields at least one rule, so the notebook is robust on the small sample.


In [4]:
freq = pd.DataFrame()
rules = pd.DataFrame()
for support in [0.20, 0.15, 0.10, 0.05, 0.03]:
    freq = apriori(baskets.astype(bool), min_support=support, use_colnames=True)
    if freq.empty:
        continue
    candidate = association_rules(freq, metric='lift', min_threshold=1.0)
    if not candidate.empty:
        rules = candidate.sort_values('lift', ascending=False)
        print('using support =', support, 'rules =', len(rules))
        break
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15)


using support = 0.2 rules = 12


,antecedents,consequents,support,confidence,lift
10,(Sports and travel),(Home and lifestyle),0.239544,0.480916,1.081033
11,(Home and lifestyle),(Sports and travel),0.239544,0.538462,1.081033
0,(Fashion accessories),(Electronic accessories),0.254753,0.503759,1.068457
1,(Electronic accessories),(Fashion accessories),0.254753,0.540323,1.068457
6,(Health and beauty),(Fashion accessories),0.212928,0.533333,1.054637
7,(Fashion accessories),(Health and beauty),0.212928,0.421053,1.054637
4,(Fashion accessories),(Food and beverages),0.243346,0.481203,1.045921
5,(Food and beverages),(Fashion accessories),0.243346,0.528926,1.045921
2,(Sports and travel),(Electronic accessories),0.239544,0.480916,1.020007
3,(Electronic accessories),(Sports and travel),0.239544,0.508065,1.020007


## Persist rules


In [5]:
out = rules.copy() if not rules.empty else pd.DataFrame(columns=['antecedents', 'consequents', 'support', 'confidence', 'lift'])
if not out.empty:
    out['antecedents'] = out['antecedents'].apply(lambda s: ', '.join(sorted(s)))
    out['consequents'] = out['consequents'].apply(lambda s: ', '.join(sorted(s)))
out[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_parquet(config.RULES_PARQUET, index=False)
print('saved:', config.RULES_PARQUET)


saved: D:\ZE5 PORTOFOLIO DS\Retail Analytics And Forecasting Platform\data\processed\rules.parquet
